# HNSW: approximate nearest neighbour search

Runs [hnsw.py](hnsw.py), a Python implementation of Hierarchical Navigable Small World graphs from
Malkov and Yashunin, [Efficient and robust approximate nearest neighbor search using Hierarchical Navigable
Small World graphs](https://arxiv.org/abs/1603.09320), on synthetic data.

- **Layers.** Every element gets a random top layer, so each layer holds about 1/M of the layer below and
  layer 0 holds them all. Each layer is a graph linking elements to nearby ones.
- **Search.** Walk greedily from the entry point on the top layer towards the query, always to a closer
  neighbour. Where no neighbour is closer, drop a layer and go on from there. On layer 0, keep the `ef`
  closest found so far and expand them nearest first, so the walk can back out of a dead end.
- **Insert.** Search for the new element as if it were a query, then link it to neighbours chosen among the
  `efConstruction` closest found, on each of its layers.
- **Choosing the neighbours.** Either the M nearest, or the paper's heuristic: take candidates nearest first and keep one only if
  it's closer to the new element than to every one already kept. Links then spread out in all directions.

The heuristic's rule is that of the relative neighbourhood graph, a subgraph of the Delaunay triangulation
in [triangulation](../triangulation), which this notebook compares against.

In [ ]:
import sys
try:
    # In Google Colab only the notebook is here: fetch the scripts it imports from GitHub
    import google.colab
    import urllib.request
    for path in ('hnsw/hnsw.py', 'triangulation/divide_and_conquer.py'):
        urllib.request.urlretrieve(
            f'https://raw.githubusercontent.com/tinkerersnotes/tinkerersnotes/main/{path}', path.split('/')[1])
except ImportError:
    sys.path.append('../triangulation')  # divide_and_conquer.py, for the Delaunay comparison

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.collections import LineCollection
from matplotlib.ticker import FuncFormatter

from hnsw import HNSW, exact_knn, recall

SEED = 1
K = 10  # neighbours per query

# Colours: categorical series in a fixed order, then muted inks for the scaffolding
BLUE, ORANGE, AQUA, YELLOW = '#2a78d6', '#eb6834', '#1baf7a', '#eda100'
INK, MUTED, EDGE, GRID = '#0b0b0b', '#898781', '#c3c2b7', '#e1e0d9'
plt.rcParams.update({
    'axes.edgecolor': EDGE, 'axes.labelcolor': INK, 'xtick.color': MUTED, 'ytick.color': MUTED,
    'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.6, 'axes.spines.top': False,
    'axes.spines.right': False, 'legend.frameon': False, 'lines.linewidth': 2,
})

def timed_build(data, **options):
    index = HNSW(data.shape[1], seed=SEED, **options)
    start = time.perf_counter()
    index.add_items(data)
    print(f'{len(data)} vectors of {data.shape[1]} dimensions, {options or "default options"}: '
          f'built in {time.perf_counter() - start:.1f} s, layer sizes '
          f'{[len(index.layer_elements(layer)) for layer in range(index.top + 1)]}')
    return index

def sweep(index, queries, truth, efs=(10, 15, 20, 30, 40, 60, 80, 120, 160)):
    """Recall and distances computed per query for each ef."""
    rows = []
    for ef in efs:
        index.distance_count = 0
        start = time.perf_counter()
        found, _ = index.search_many(queries, K, ef)
        per_query = (time.perf_counter() - start) / len(queries)
        rows.append((ef, recall(found, truth), index.distance_count / len(queries), per_query))
        print(f'ef {ef:4}: recall@{K} {rows[-1][1]:.3f}, {rows[-1][2]:6.0f} distances, '
              f'{1000 * per_query:.2f} ms per query')
    return np.array(rows)

def plain_numbers(axis):
    """Log axis ticks as plain numbers, 1,000 rather than 10^3."""
    axis.set_major_formatter(FuncFormatter(lambda x, _: f'{x:,.0f}'))
    axis.set_minor_formatter(FuncFormatter(lambda x, _: f'{x:,.0f}' if str(int(x))[0] in '25' else ''))

## The layers, in 2D

2000 points uniform in the unit square, with the paper's defaults: M = 16 (32 links on layer 0),
`efConstruction` = 100, mL = 1/ln M. One query is traced through the layers. On the upper layers
the walk is purely greedy (`ef` = 1), so it follows a single path. On layer 0, with `ef` = 10, it expands
the elements drawn in blue, nearest first, and computes distances to the visited ones, in orange.

In [ ]:
points = np.random.default_rng(SEED).uniform(size=(2000, 2))
index = timed_build(points)

query = np.array([0.62, 0.38])
trace = []
index.distance_count = 0
ids, dist = index.search(query, K, ef=K, trace=trace)
truth, _ = exact_knn(points, query[None], K)
print(f'Found {sorted(ids.tolist())}\nExact {sorted(truth[0].tolist())}\n'
      f'{index.distance_count} distances computed, {len(points)} for brute force')

fig, axes = plt.subplots(1, len(trace), figsize=(5 * len(trace), 5.2))
for ax, step in zip(axes, trace):
    layer = step['layer']
    members = index.layer_elements(layer)
    ax.add_collection(LineCollection(points[index.edges(layer)], colors=EDGE, linewidths=0.5, zorder=1))
    ax.scatter(*points[members].T, s=6 if layer == 0 else 16, color=MUTED, zorder=2)
    visited = list(step['visited'])
    ax.scatter(*points[visited].T, s=24, color=ORANGE, zorder=3, label='visited')
    path = points[step['expanded']]
    ax.plot(*path.T, '-o', color=BLUE, markersize=5, zorder=4, label='expanded, in order')
    if layer == 0:
        ax.scatter(*points[ids].T, s=90, facecolors='none', edgecolors=AQUA, linewidths=2, zorder=5,
                   label=f'{K} nearest found')
    ax.scatter(*query, marker='*', s=220, color=INK, zorder=6, label='query')
    ax.set_title(f'Layer {layer}: {len(members)} elements, {len(visited)} visited', color=INK)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect('equal')
    ax.grid(False)
axes[-1].legend(loc='upper left', bbox_to_anchor=(1.02, 1))
plt.tight_layout()
plt.show()

Element counts per layer follow the exponential level distribution: layer l should hold about
n M<sup>−l</sup> of them.

In [ ]:
for layer in range(index.top + 1):
    print(f'layer {layer}: {len(index.layer_elements(layer)):5} elements, expected '
          f'{len(points) * index.M ** -layer:7.1f}, mean degree {index.degrees(layer).mean():.1f}')

## Layer 0 and the Delaunay triangulation

The paper motivates the heuristic with Delaunay graphs: on a Delaunay graph, a greedy walk always reaches the
true nearest neighbour. Here layer 0 is compared with the Delaunay triangulation of the same points, from
[divide_and_conquer.py](../triangulation/divide_and_conquer.py), and with its subgraph, the relative
neighbourhood graph (RNG): an edge ab is in the RNG when no other point is closer to both a and b than they are
to each other.

Three kinds of edge are compared: the heuristic's, the M nearest's, and Delaunay or RNG edges that layer 0
misses.

In [ ]:
from divide_and_conquer import Triangulation

delaunay = {(min(a, b), max(a, b))
            for triangle in Triangulation(points).triangles()
            for a, b in zip(triangle, triangle[1:] + triangle[:1])}

def in_rng(a, b):
    """No third point closer to both a and b than they are to each other."""
    d = np.sum((points[a] - points[b]) ** 2)
    farther = np.maximum(np.sum((points - points[a]) ** 2, 1), np.sum((points - points[b]) ** 2, 1))
    farther[[a, b]] = np.inf
    return not (farther < d).any()

rng_edges = {e for e in delaunay if in_rng(*e)}  # The RNG is a subgraph of the Delaunay graph
print(f'Delaunay: {len(delaunay)} edges, RNG: {len(rng_edges)}')

nearest = timed_build(points, heuristic=False)
graphs = {'Heuristic (alg. 4)': index, 'M nearest (alg. 3)': nearest}
for name, g in graphs.items():
    edges = set(g.edges(0))
    print(f'{name}: {len(edges)} edges on layer 0, {len(edges & delaunay) / len(edges):.0%} of them Delaunay; '
          f'they hold {len(edges & delaunay) / len(delaunay):.0%} of the Delaunay edges and '
          f'{len(edges & rng_edges) / len(rng_edges):.0%} of the RNG')

window = 0.35, 0.65  # zoom in on the middle of the square
fig, axes = plt.subplots(1, 2, figsize=(12, 6.2))
for ax, (name, g) in zip(axes, graphs.items()):
    edges = set(g.edges(0))
    kinds = [(edges - delaunay, ORANGE, 0.5, 'not Delaunay'),  # thinnest, underneath
             (edges & delaunay - rng_edges, AQUA, 1.2, 'Delaunay, not RNG'),
             (edges & rng_edges, BLUE, 1.6, 'in the RNG')]
    for subset, colour, width, label in kinds:
        ax.add_collection(LineCollection(points[sorted(subset)], colors=colour, linewidths=width,
                                         label=f'{label}: {len(subset)}'))
    ax.scatter(*points.T, s=10, color=INK, zorder=3)
    ax.set_title(f'{name}: layer 0', color=INK)
    ax.set_xlim(*window)
    ax.set_ylim(*window)
    ax.set_aspect('equal')
    ax.grid(False)
    ax.legend(loc='upper left', bbox_to_anchor=(0, -0.04), ncol=3, fontsize=9)
plt.tight_layout()
plt.show()

The heuristic keeps every RNG edge and about as many edges as the Delaunay graph has, but not quite the same
ones. Links are chosen when an element is inserted, among the elements already there. The first elements had
few others around, so they were linked across long distances. Later elements filled the gaps, and those links
stopped being Delaunay edges, but nothing removes a link until its element has more than 2M. So the long links
stay, as shortcuts across the square. The M nearest instead link each point to everything around it: over four
times as many edges, bunched in the same directions.

## 16 dimensions: recall against work

10,000 vectors with 16 independent normal coordinates, and 200 queries from the same distribution. Recall@10
is the share of the true 10 nearest neighbours found, checked against brute force. Raising `ef` buys recall
with more distance computations.

In [ ]:
rng = np.random.default_rng(SEED)
data = rng.normal(size=(10_000, 16))
queries = rng.normal(size=(200, 16))
truth, _ = exact_knn(data, queries, K)
index16 = timed_build(data)
gaussian = sweep(index16, queries, truth)

start = time.perf_counter()
exact_knn(data, queries, K)
print(f'Brute force with numpy: {1000 * (time.perf_counter() - start) / len(queries):.2f} ms per query, '
      f'{len(data)} distances')

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(gaussian[:, 2], gaussian[:, 1], '-o', color=BLUE, markersize=6)
for ef, r, d, _ in gaussian[[0, 2, 4, 6, 8]]:
    ax.annotate(f'ef {ef:.0f}', (d, r), textcoords='offset points', xytext=(6, -12), color=MUTED, fontsize=9)
ax.set_xscale('log')
plain_numbers(ax.xaxis)
ax.set_xlabel('Distances computed per query (brute force: 10,000)')
ax.set_ylabel(f'Recall@{K}')
ax.set_title('HNSW on 10,000 Gaussian vectors in 16 dimensions', color=INK)
plt.show()

HNSW computes a small fraction of the distances, yet numpy's brute force is still faster per query here:
it does all 10,000 at once in compiled code, while this implementation pays Python's overhead on every step
of the walk. Compiled HNSW libraries do the same walk in microseconds; the count of distances is what carries
over between implementations.

## Growing the index

HNSW is built one insertion at a time, so one index can be measured at several sizes as it grows: 8 dimensions
up to 20,000 vectors, with `ef` = 20. Brute force grows linearly; the walk's cost grows roughly with the
logarithm of the size.

In [ ]:
rng = np.random.default_rng(SEED + 1)
data8 = rng.normal(size=(20_000, 8))
queries8 = rng.normal(size=(200, 8))
growing = HNSW(8, seed=SEED)
sizes = [500, 1000, 2000, 5000, 10_000, 20_000]
work = []
for n in sizes:
    growing.add_items(data8[len(growing):n])
    truth8, _ = exact_knn(data8[:n], queries8, K)
    growing.distance_count = 0
    found, _ = growing.search_many(queries8, K, ef=20)
    work.append(growing.distance_count / len(queries8))
    print(f'{n:6} vectors: recall@{K} {recall(found, truth8):.3f}, {work[-1]:5.0f} distances per query, '
          f'{growing.top + 1} layers')

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(sizes, sizes, '--', color=MUTED, linewidth=1.5, label='brute force')
ax.plot(sizes, work, '-o', color=BLUE, markersize=6, label='HNSW, ef 20')
ax.set_xscale('log')
ax.set_yscale('log')
plain_numbers(ax.xaxis)
plain_numbers(ax.yaxis)
ax.set_xlabel('Vectors in the index')
ax.set_ylabel('Distances computed per query')
ax.set_title('Work per query as the index grows, 8 dimensions', color=INK)
ax.legend()
plt.show()

## Clustered data: why the heuristic matters

The paper's Fig. 7 setting: 100 tight, well separated clusters in 10 dimensions, with M = 6. The paper reports
that with the M nearest as links, "the search stucks at the clusters boundaries". Two effects show up here:

- **Unreachable elements.** An element whose links overflow keeps only its 2M nearest. In a crowded cluster,
  everyone's nearest are the same few central elements, so links to the outer ones are dropped from every side.
  Some elements end up with no path from the entry point on layer 0 at all.
- **The wrong cluster.** Links between clusters are rarely among anyone's nearest, so a greedy walk can descend
  into a cluster that isn't the query's and never leave it.

The heuristic keeps a link in a new direction whenever no element already kept is closer to it, both on insertion
and when pruning. Far fewer elements get cut off, and links between clusters survive.

In [ ]:
rng = np.random.default_rng(SEED)
centres = rng.uniform(-50, 50, size=(100, 10))
labels, query_labels = rng.integers(100, size=10_000), rng.integers(100, size=200)
clustered = centres[labels] + rng.normal(size=(10_000, 10))
cluster_queries = centres[query_labels] + rng.normal(size=(200, 10))
cluster_truth, _ = exact_knn(clustered, cluster_queries, K)

def reachable(index):
    """Elements with a path from the entry point on layer 0."""
    seen, stack = {index.entry}, [index.entry]
    while stack:
        for j in index.links[stack.pop()][0]:
            if j not in seen:
                seen.add(j)
                stack.append(j)
    return seen

efs = (10, 20, 40, 80, 160)
results = {}
for name, options in {'M nearest (alg. 3)': dict(heuristic=False), 'Heuristic (alg. 4)': {}}.items():
    print(name)
    g = timed_build(clustered, M=6, **options)
    results[name] = sweep(g, cluster_queries, cluster_truth, efs)
    found, _ = g.search_many(cluster_queries, K, ef=40)
    wrong = np.mean(labels[found[:, 0]] != query_labels)
    print(f'{len(g) - len(reachable(g))} elements unreachable on layer 0; with ef 40, '
          f'{wrong:.1%} of the queries end in another cluster\n')

fig, ax = plt.subplots(figsize=(7, 4.5))
for (name, rows), colour in zip(results.items(), (ORANGE, BLUE)):
    ax.plot(rows[:, 2], rows[:, 1], '-o', color=colour, markersize=6, label=name)
ax.set_xscale('log')
ax.set_ylim(0, 1.02)
plain_numbers(ax.xaxis)
ax.set_xlabel('Distances computed per query')
ax.set_ylabel(f'Recall@{K}')
ax.set_title('100 clusters in 10 dimensions, M = 6', color=INK)
ax.legend(loc='lower right')
plt.show()

## Cosine distance

Text embeddings are usually compared by the angle between them. With `metric='cosine'` the index normalises
every vector once and uses 1 − cos as the distance. Here 5000 vectors in 64 dimensions stand in for
embeddings: each is a mix of a few of 50 random "topic" directions plus noise, so they share structure the way
embeddings of related texts do.

In [ ]:
rng = np.random.default_rng(SEED)
topics = rng.normal(size=(50, 64))

def documents(n):
    """Each vector: three topics with random weights, plus noise."""
    weights = np.zeros((n, len(topics)))
    for row in weights:
        row[rng.choice(len(topics), 3, replace=False)] = rng.exponential(size=3)
    return weights @ topics + 0.3 * rng.normal(size=(n, 64))

docs, doc_queries = documents(5000), documents(200)
doc_truth, _ = exact_knn(docs, doc_queries, K, metric='cosine')
cosine = sweep(timed_build(docs, metric='cosine'), doc_queries, doc_truth, efs=(10, 20, 40, 80))

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tinkerersnotes/tinkerersnotes/blob/main/hnsw/hnsw.ipynb)